## 🎯 Learning Objectives
* Understand the fundamental concept of tool calling in modern LangChain agents.
* Learn how to define and integrate custom tools into a LangChain agent.
* Implement a basic LangChain agent capable of using tools to extend its capabilities.
* Analyze the execution flow of an agent utilizing tools and interpret its output.
* Identify common use cases and performance considerations for tool-augmented agents.


## Tool Calling with Modern LangChain Agents

In the rapidly evolving landscape of Agentic AI, Large Language Models (LLMs) are no longer confined to just generating text. To truly build intelligent, autonomous agents, LLMs need to interact with the real world, perform calculations, access up-to-date information, and execute actions. This is where **tool calling** comes into play.

### The Analogy: A Master Craftsman with Specialized Tools

Imagine a master craftsman who is incredibly skilled at designing and planning complex projects. However, this craftsman isn't necessarily the best at every single task – they might not be the fastest at cutting wood, the most precise at welding, or the most knowledgeable about the latest material prices. Instead, they have a workshop full of specialized tools and skilled apprentices (tools) they can delegate specific tasks to.

*   **The Master Craftsman (LLM):** The LLM is the brain, responsible for understanding the user's request, planning the overall approach, and deciding *when* and *which* specialized task needs to be performed.
*   **The Specialized Tools (Functions/APIs):** These are external functions, APIs, or custom code snippets designed to perform specific, well-defined tasks that an LLM might struggle with (e.g., complex math, real-time data retrieval, interacting with external systems).
*   **Delegation (Tool Calling):** When the craftsman encounters a task best handled by a tool, they instruct the tool, provide the necessary inputs, and wait for the result. The LLM does the same: it identifies a need, formats a call to a specific tool with appropriate arguments, and processes the tool's output.

### Why are Tools Crucial for LLMs?

1.  **Overcoming LLM Limitations:** LLMs are powerful pattern matchers but have inherent limitations:
    *   **Factual Recall:** Their knowledge is limited to their training data cutoff.
    *   **Real-time Data:** They cannot access current events, stock prices, or weather.
    *   **Complex Calculations:** They are prone to errors in arithmetic or logical reasoning.
    *   **External Actions:** They cannot directly send emails, book flights, or interact with databases.
2.  **Expanding Capabilities:** Tools allow LLMs to:
    *   **Access up-to-date information:** Via search engines, databases, or APIs.
    *   **Perform precise calculations:** Using calculators or data analysis libraries.
    *   **Interact with external systems:** Through APIs for CRM, ERP, email, etc.
    *   **Execute code:** For data manipulation, scripting, or automation.
3.  **Enhanced Reliability and Accuracy:** By delegating specific tasks to reliable, deterministic tools, the overall accuracy and trustworthiness of the agent's responses improve.

### Modern LangChain's Approach to Tool Calling

LangChain, as of 2026, has significantly streamlined its agent architecture, moving towards more robust and intuitive patterns for tool calling. The `create_tool_calling_agent` function (or similar factory functions for specific LLM providers) is a prime example. This approach leverages the native tool-calling capabilities of modern LLMs (like OpenAI's function calling or Anthropic's tool use), allowing the LLM itself to decide when and how to invoke a tool based on its understanding of the prompt and the available tools.

**Core Components:**

*   **LLM:** The brain of the agent, typically a chat model with native tool-calling support.
*   **Tools:** Python functions decorated with `@tool` or `BaseTool` instances, describing their purpose and expected inputs.
*   **Prompt:** A `ChatPromptTemplate` that guides the LLM, often including a placeholder for tool instructions.
*   **Agent:** The `Runnable` object created by `create_tool_calling_agent` that orchestrates the LLM and tools.
*   **AgentExecutor:** The runtime environment that executes the agent, managing the conversational loop, tool calls, and final response generation.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install langchain langchain-openai

import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# --- 1. Define Tools ---
# Tools are Python functions that the LLM can call. They should be self-contained
# and perform a specific task. We use the @tool decorator for simplicity.

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location. Use this tool for any weather-related questions.
    The location should be a city name, e.g., 'San Francisco'.
    """
    # In a real-world scenario, this would call an external weather API.
    # For this example, we'll return a mock response.
    if "san francisco" in location.lower():
        return "It's 18 degrees Celsius and sunny in San Francisco."
    elif "new york" in location.lower():
        return "It's 10 degrees Celsius and cloudy in New York."
    elif "london" in location.lower():
        return "It's 8 degrees Celsius and rainy in London."
    else:
        return f"Sorry, I don't have weather information for {location}."

@tool
def calculate_sum(a: float, b: float) -> float:
    """Calculates the sum of two numbers. Use this tool for any addition operations.
    Input should be two floating-point numbers.
    """
    return a + b

# List of all tools available to the agent
tools = [get_current_weather, calculate_sum]

# --- 2. Initialize the Language Model (LLM) ---
# We'll use a modern OpenAI chat model that supports tool calling natively.
# Ensure you have your OPENAI_API_KEY set in your environment variables.
# For 2026, gpt-4o is a strong choice for its multimodal and tool-calling capabilities.
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- 3. Create the Prompt Template ---
# The prompt guides the LLM. It's crucial to include a MessagesPlaceholder for `agent_scratchpad`
# which the agent uses to inject tool calls and their outputs.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. You have access to tools to help you answer questions."),
    MessagesPlaceholder("chat_history"), # For conversational memory, if needed
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"), # Essential for tool calling orchestration
])

# --- 4. Create the Agent ---
# `create_tool_calling_agent` automatically formats the tools and prompt for the LLM
# to understand and use its native tool-calling capabilities.
agent = create_tool_calling_agent(llm, tools, prompt)

# --- 5. Create the Agent Executor ---
# The AgentExecutor is the runtime that takes the agent and tools and executes the chain.
# It handles the loop of LLM thinking, tool calling, and processing results.
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# --- 6. Run the Agent with various queries ---
print("\n--- Query 1: Weather information ---")
response1 = agent_executor.invoke({"input": "What's the weather like in San Francisco?"})
print(f"Agent's final answer: {response1['output']}")

print("\n--- Query 2: Calculation ---")
response2 = agent_executor.invoke({"input": "What is 123.45 plus 67.89?"})
print(f"Agent's final answer: {response2['output']}")

print("\n--- Query 3: General knowledge (no tool needed) ---")
response3 = agent_executor.invoke({"input": "What is the capital of France?"})
print(f"Agent's final answer: {response3['output']}")

print("\n--- Query 4: Tool not available ---")
response4 = agent_executor.invoke({"input": "Can you tell me the current stock price of Google?"})
print(f"Agent's final answer: {response4['output']}")

# Example with chat history (though not fully implemented in this simple prompt)
# chat_history = [
#     HumanMessage(content="Hi, what's up?"),
#     AIMessage(content="Hello! How can I help you today?")
# ]
# response_with_history = agent_executor.invoke({"input": "What's the weather like in London?", "chat_history": chat_history})
# print(f"\nAgent's final answer with history: {response_with_history['output']}")


### Interpreting the Code Output and Performance Trade-offs

When you run the code, pay close attention to the `verbose=True` output from the `AgentExecutor`. This detailed log reveals the agent's thought process:

1.  **`> Entering new AgentExecutor chain...`**: The agent starts processing the input.
2.  **`Thought:`**: The LLM's internal monologue. It analyzes the user's query and decides if a tool is needed. If so, it identifies *which* tool and *what arguments* to pass to it.
3.  **`Calling Tool: <tool_name> with input: <tool_arguments>`**: The agent executes the chosen tool with the specified inputs.
4.  **`Tool Output: <tool_result>`**: The result returned by the tool is fed back to the LLM.
5.  **`Thought:`**: The LLM processes the tool's output and formulates a final, human-readable answer.
6.  **`Final Answer:`**: The agent's ultimate response to the user.

For queries that don't require a tool (e.g., "What is the capital of France?"), the LLM directly provides the `Final Answer` without any tool calls.

For queries where a tool *might* be useful but isn't available (e.g., "stock price"), the LLM will typically state that it cannot fulfill the request, demonstrating its awareness of its limitations and available tools.

### Performance Trade-offs

While incredibly powerful, tool calling introduces several performance considerations:

*   **Latency:** Each tool call involves an additional round trip: LLM inference -> tool execution -> LLM inference. This adds latency compared to a single LLM call. Complex tasks requiring multiple tool calls will accumulate more latency.
*   **Cost:** Each LLM inference step (the initial decision, and then processing the tool output) incurs cost. More tool calls mean more LLM tokens consumed, leading to higher operational costs.
*   **Reliability:** The overall reliability of your agent becomes dependent on the reliability of its tools. If a tool fails, returns incorrect data, or is slow, it directly impacts the agent's performance and user experience.
*   **Complexity:** Managing a growing suite of tools, their documentation, and potential conflicts can add complexity to agent development and maintenance.

### Typical Use Cases for Tool-Augmented Agents

Tool calling unlocks a vast array of applications for LLM agents:

*   **Data Retrieval:** Accessing real-time information (weather, news, stock prices), querying databases, or searching internal documents.
*   **Action Execution:** Sending emails, creating calendar events, updating CRM records, booking appointments, or controlling IoT devices.
*   **Complex Calculations & Analysis:** Performing financial modeling, statistical analysis, data aggregation, or unit conversions.
*   **Code Generation & Execution:** Writing and running code snippets to solve problems, debug, or automate tasks.
*   **Multi-modal Interactions:** Generating images, processing audio, or interacting with visual data through specialized tools.
*   **Personal Assistants:** Combining all the above to provide comprehensive support, from scheduling to information retrieval and task automation.


### Resources

*   **LangChain Agents Documentation:** [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)
*   **LangChain Tools Documentation:** [https://python.langchain.com/docs/modules/tools/](https://python.langchain.com/docs/modules/tools/)
*   **OpenAI Function Calling (Tool Use) Guide:** [https://platform.openai.com/docs/guides/function-calling](https://platform.openai.com/docs/guides/function-calling)
*   **Anthropic Tool Use Documentation:** [https://docs.anthropic.com/claude/docs/tool-use](https://docs.anthropic.com/claude/docs/tool-use)
*   **LangChain Expression Language (LCEL) for Agents:** [https://python.langchain.com/docs/expression_language/how_to/agent](https://python.langchain.com/docs/expression_language/how_to/agent)
